# 신용카드 사기 탐지 프로젝트 (Credit Card Fraud Detection)

## 1. 데이터 로딩 및 전처리

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("creditcard.csv")

# Amount 정규화
scaler = StandardScaler()
df["Amount"] = scaler.fit_transform(df[["Amount"]])

# Time 컬럼 제거
df.drop(columns=["Time"], inplace=True)

# 학습/검증 분할
X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


## 2. 불균형 데이터 처리: 언더샘플링 & SMOTE

In [3]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# 언더샘플링
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)

# 오버샘플링 (SMOTE)
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("언더샘플링 후:", pd.Series(y_rus).value_counts().to_dict())
print("SMOTE 후:", pd.Series(y_smote).value_counts().to_dict())


C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physica

언더샘플링 후: {0: 394, 1: 394}
SMOTE 후: {0: 227451, 1: 227451}


## 3. 기본 모델: 로지스틱 회귀, 결정 트리

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier()
}

for name, model in models.items():
    model.fit(X_rus, y_rus)
    probas = model.predict_proba(X_val)[:,1]
    roc = roc_auc_score(y_val, probas)
    prec, rec, _ = precision_recall_curve(y_val, probas)
    pr_auc = auc(rec, prec)
    print(f"\n{name}")
    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR AUC: {pr_auc:.4f}")



Logistic Regression
ROC-AUC: 0.9757
PR AUC: 0.7143

Decision Tree
ROC-AUC: 0.9103
PR AUC: 0.4672


## 4. 고급 모델: Random Forest, XGBoost, Isolation Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier

models_adv = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss'),
    "Isolation Forest": IsolationForest(contamination=0.001, random_state=42)
}

for name, model in models_adv.items():
    if name == "Isolation Forest":
        model.fit(X_train)
        preds = model.predict(X_val)
        preds = (preds == -1).astype(int)
        probas = preds
    else:
        model.fit(X_smote, y_smote)
        probas = model.predict_proba(X_val)[:,1]
    roc = roc_auc_score(y_val, probas)
    prec, rec, _ = precision_recall_curve(y_val, probas)
    pr_auc = auc(rec, prec)
    print(f"\n{name}")
    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR AUC: {pr_auc:.4f}")



Random Forest
ROC-AUC: 0.9737
PR AUC: 0.8746


C:\Users\shjun\AppData\Roaming\Python\Python310\site-packages\xgboost\training.py:183: UserWarning: [16:16:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost
ROC-AUC: 0.9753
PR AUC: 0.8736

Isolation Forest
ROC-AUC: 0.6170
PR AUC: 0.2949


## 5. 딥러닝: Autoencoder 기반 이상 탐지

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_train.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded)
decoded = Dense(16, activation='relu')(encoded)
output = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_train, X_train, epochs=10, batch_size=256, shuffle=True, validation_split=0.1, verbose=0)

# Reconstruction error
X_val_pred = autoencoder.predict(X_val)
mse = ((X_val - X_val_pred)**2).mean(axis=1)
roc = roc_auc_score(y_val, mse)
prec, rec, _ = precision_recall_curve(y_val, mse)
pr_auc = auc(rec, prec)

print(f"\nAutoencoder")
print(f"ROC-AUC: {roc:.4f}")
print(f"PR AUC: {pr_auc:.4f}")


1781/1781 [==============================] - 3s 2ms/step

Autoencoder
ROC-AUC: 0.9312
PR AUC: 0.0634
